In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import wandb

import src.data_preprocessing.caption_generation as cg
import src.data_preprocessing.create_aux_data as cad
import src.data_preprocessing.data_utils as du
import src.data_preprocessing.gee_utils as gu
from src.data.base_caption_builder import BaseCaptionBuilder, DummyCaptionBuilder
from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

# Prediction

In [ ]:
api = wandb.Api()
runs = api.runs("aether_xai/s2bms_prediction")

rows = []
for run in runs:
    row = {"run_id": run.id, "run_name": run.name, **run.config, **run.summary._json_dict}
    rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
# Prediction:
from collections import defaultdict

dict_check = defaultdict(list)


attr_cols = [
    "loss",
    "lr",
    "geo_encoder",
    "resnet_v",
    "pretrained",
    "concat_name_encoders",
    "run_name",
    "run_id",
    "geo_data",
    "pred_head",
    "n_modalities",
    "mod_1",
    "size_1",
    "batch_size",
    "pred_head_h_dim",
]
metric_cols = ["test_top10", "test_top5", "test_mse"]

new_df = {x: [] for x in attr_cols + metric_cols + ["seed"]}

for row in df.itertuples():

    loss = row.model["loss_fn"]["_target_"]
    lr = row.model["optimizer"]["lr"]
    geo_encoder = row.model["geo_encoder"]["_target_"]
    if geo_encoder == "src.models.components.geo_encoders.cnn_encoder.CNNEncoder":
        resnet_v = row.model["geo_encoder"].get("resnet_version")
        pretrained = row.model["geo_encoder"].get("pretrained_cnn")
    else:
        resnet_v = None
        pretrained = None

    if geo_encoder == "src.models.components.geo_encoders.encoder_wrapper.EncoderWrapper":
        concat_name_encoders = "_".join(
            [
                x["encoder"]["geo_data_name"] + "-" + x["encoder"]["_target_"]
                for x in row.model["geo_encoder"]["encoder_branches"]
            ]
        )
    else:
        concat_name_encoders = None

    geo_data = row.model["geo_encoder"].get("geo_data_name")
    if geo_data == "s2":  # many duplicates
        continue
    elif (
        geo_data == "aef_avr"
    ):  # skipping because only 1 test run exist, use aef + average encoder instead
        continue
    elif pd.isna(geo_data):  # skipping fused for now
        continue

    pred_head = row.model["prediction_head"]["_target_"]
    pred_head_h_dim = row.model["prediction_head"].get("hidden_dim")
    seed = row.seed

    n_modalities = len(row.data["dataset"]["modalities"])
    mod_1 = list(row.data["dataset"]["modalities"].keys())[0]
    if row.data["dataset"]["modalities"][mod_1] is None:
        size_1 = None
    else:
        size_1 = row.data["dataset"]["modalities"][mod_1].get("size")
    batch_size = row.data["batch_size"]

    tuple_key = (
        loss,
        lr,
        geo_encoder,
        resnet_v,
        pretrained,
        concat_name_encoders,
        geo_data,
        pred_head,
        seed,
        n_modalities,
        mod_1,
        size_1,
        batch_size,
        pred_head_h_dim,
    )
    if tuple_key in dict_check:
        print(f"Duplicate found for {tuple_key}")

    dict_check[tuple_key].append(row.run_id)

    test_top10 = row.test_top_10_acc
    test_top5 = row.test_top_5_acc
    test_mse = row.test_mse_loss

    new_df["run_name"].append(row.run_name)
    new_df["run_id"].append(row.run_id)
    new_df["loss"].append(loss)
    new_df["lr"].append(lr)
    new_df["geo_encoder"].append(geo_encoder)
    new_df["resnet_v"].append(resnet_v)
    new_df["pretrained"].append(pretrained)
    new_df["concat_name_encoders"].append(concat_name_encoders)
    new_df["geo_data"].append(geo_data)
    new_df["pred_head"].append(pred_head)
    new_df["seed"].append(seed)
    new_df["n_modalities"].append(n_modalities)
    new_df["mod_1"].append(mod_1)
    new_df["size_1"].append(size_1)
    new_df["batch_size"].append(batch_size)
    new_df["pred_head_h_dim"].append(pred_head_h_dim)
    new_df["test_top10"].append(test_top10)
    new_df["test_top5"].append(test_top5)
    new_df["test_mse"].append(test_mse)

new_df = pd.DataFrame(new_df)

# average across seeds, but first check if any combination of hyperparameters has less than 3 seeds:
seed_counts = new_df.groupby(attr_cols, dropna=False).count()
# seed_counts[seed_counts['seed'] < 3]
seed_counts

cols_drop = ["loss", "lr", "pretrained", "mod_1", "n_modalities"]  # , 'pred_head_h_dim']
new_df = new_df.drop(columns=cols_drop)
attr_cols = [x for x in attr_cols if x not in cols_drop]

for c in metric_cols:
    new_df[c] = new_df[c] * 100

mean_scores = (
    new_df.groupby(attr_cols, dropna=False)
    .agg(
        test_top10=("test_top10", "mean"),
        test_top10_sem=("test_top10", "sem"),
        test_top5=("test_top5", "mean"),
        test_top5_sem=("test_top5", "sem"),
        test_mse=("test_mse", "mean"),
        test_mse_sem=("test_mse", "sem"),
        seed=("seed", "mean"),
    )
    .reset_index()
)
mean_scores = mean_scores.sort_values(
    by=["test_mse", "test_top10", "test_top5"], ascending=[True, False, False]
)
mean_scores

new_df = new_df[new_df["geo_data"].isin(["aef"])]
new_df = new_df[
    new_df["pred_head"] == "src.models.components.pred_heads.mlp_pred_head.MLPPredictionHead"
]
new_df = new_df[
    new_df["geo_encoder"].isin(
        ["src.models.components.geo_encoders.average_encoder.AverageEncoder"]
    )
]
new_df = new_df[new_df["size_1"].isin([128])]
new_df

In [ ]:
mean_scores = mean_scores[mean_scores["resnet_v"] != 50]
# mean_scores = mean_scores[mean_scores['pred_head'] != 'src.models.components.pred_heads.linear_pred_head.LinearPredictionHead']
for c in mean_scores.columns:
    if "top" in c:
        mean_scores[c] = mean_scores[c].round(1)
    elif "mse" in c:
        mean_scores[c] = mean_scores[c].round(2)
mean_scores

In [ ]:
new_df.groupby(["resnet_v"], dropna=False).count()

In [ ]:
row

In [ ]:
model_names = dict_check[
    (
        "src.models.components.loss_fns.bce_loss.BCELoss",
        0.01,
        "src.models.components.geo_encoders.encoder_wrapper.EncoderWrapper",
        None,
        None,
        None,
        "src.models.components.pred_heads.mlp_pred_head.MLPPredictionHead",
        654,
        2,
        "coords",
        None,
        128,
        512,
    )
]
df[df.run_id.isin(model_names)].model.iloc[1]

# Alignment

In [ ]:
api = wandb.Api()
runs = api.runs("aether_xai/s2bms_alignment")

rows = []
for run in runs:
    row = {"run_id": run.id, "run_name": run.name, **run.config, **run.summary._json_dict}
    rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
df["experiment"]

In [ ]:
# Prediction:
from collections import defaultdict

dict_check = defaultdict(list)

attr_cols = [
    "loss",
    "lr",
    "sigma",
    "temperature",
    "experiment",
    "text_encoder",
    "path",
    "run_name",
    "geo_data",
    "n_modalities",
    "mod_1",
    "size_1",
    "batch_size",
]
metric_cols = [x for x in df.columns if "test_" in x]

new_df = {x: [] for x in attr_cols + metric_cols + ["seed"]}

for row in df.iterrows():
    row = row[1]
    loss = row.model["loss_fn"]["_target_"]
    sigma = row.model["loss_fn"].get("sigma")
    temperature = row.model["loss_fn"].get("temperature")
    lr = row.model["optimizer"]["lr"]
    geo_encoder = row.model["geo_encoder"]["_target_"]
    text_encoder = row.model["text_encoder"]["_target_"]

    if geo_encoder == "src.models.components.geo_encoders.encoder_wrapper.EncoderWrapper":
        concat_name_encoders = "_".join(
            [
                x["encoder"]["geo_data_name"] + "-" + x["encoder"]["_target_"]
                for x in row.model["geo_encoder"]["encoder_branches"]
            ]
        )
    else:
        concat_name_encoders = None

    geo_data = row.model["geo_encoder"].get("geo_data_name")

    seed = row.seed

    n_modalities = len(row.data["dataset"]["modalities"])
    mod_1 = list(row.data["dataset"]["modalities"].keys())[0]
    if row.data["dataset"]["modalities"][mod_1] is None:
        size_1 = None
        path_1 = None
    else:
        size_1 = row.data["dataset"]["modalities"][mod_1].get("size")
        path_1 = row.data["dataset"]["modalities"][mod_1].get("path")
    batch_size = row.data["batch_size"]
    experiment = row.experiment

    tuple_key = (
        experiment,
        loss,
        lr,
        geo_encoder,
        concat_name_encoders,
        geo_data,
        seed,
        n_modalities,
        mod_1,
        size_1,
        batch_size,
        sigma,
        temperature,
        text_encoder,
        path_1,
    )
    if tuple_key in dict_check:
        print(f"Duplicate found for {tuple_key}")

    dict_check[tuple_key].append(row.run_id)

    # test_top10 = row.test_top_10_acc
    # test_top5 = row.test_top_5_acc
    # test_mse = row.test_mse_loss

    new_df["loss"].append(loss)
    new_df["lr"].append(lr)
    new_df["geo_data"].append(geo_data)
    new_df["seed"].append(seed)
    new_df["n_modalities"].append(n_modalities)
    new_df["mod_1"].append(mod_1)
    new_df["size_1"].append(size_1)
    new_df["batch_size"].append(batch_size)
    new_df["experiment"].append(experiment)
    new_df["sigma"].append(sigma)
    new_df["temperature"].append(temperature)
    new_df["text_encoder"].append(text_encoder)
    new_df["path"].append(path_1)
    new_df["run_name"].append(row.run_name)
    for col in metric_cols:
        new_df[col].append(row[col])
    # new_df['test_top10'].append(test_top10)
    # new_df['test_top5'].append(test_top5)
    # new_df['test_mse'].append(test_mse)

new_df = pd.DataFrame(new_df)

# average across seeds, but first check if any combination of hyperparameters has less than 3 seeds:
seed_counts = new_df.groupby(attr_cols, dropna=False).count()
# seed_counts[seed_counts['seed'] < 3]
seed_counts

cols_drop = [
    "lr",
    "mod_1",
    "n_modalities",
    "batch_size",
    "temperature",
    "text_encoder",
    "geo_data",
    "size_1",
] + [x for x in metric_cols if "index" not in x]
new_df = new_df.drop(columns=cols_drop)
attr_cols = [x for x in attr_cols if x not in cols_drop]

# for c in metric_cols:
#     new_df[c] = new_df[c] * 100

# mean_scores = (
#     new_df.groupby(attr_cols, dropna=False)
#     .agg(
#         test_top10=("test_top10", "mean"),
#         test_top10_sem=("test_top10", "sem"),
#         test_top5=("test_top5", "mean"),
#         test_top5_sem=("test_top5", "sem"),
#         test_mse=("test_mse", "mean"),
#         test_mse_sem=("test_mse", "sem"),
#         seed=("seed", "mean"),
#     )
#     .reset_index()
# )
# mean_scores = mean_scores.sort_values(by=['test_mse', 'test_top10', 'test_top5' ], ascending=[True, False, False])
# mean_scores
# new_df
new_df[new_df["experiment"].isin(["lab_avr_aef_128_mlp_cliptext", "lab_avr_aef_128_mlp_llm"])]

runs_keep = [
    "breezy-dew-64",
    "stoic-glade-65",
    "deep-plasma-67",
    "light-oath-102",
    "kind-surf-110",
]

dict_name_scores = {
    "test_avr_top-dyn_k_index": "Average",
    "test_dyn_k_index_bioclim_06_min": "Minimum temperature",
    "test_dyn_k_index_bioclim_12_max": "Precipitation",
    "test_dyn_k_index_corine_frac_11_max": "Urban fabric",
    "test_dyn_k_index_corine_frac_12_max": "Industrial areas",
    "test_dyn_k_index_corine_frac_21_max": "Arable land",
    "test_dyn_k_index_corine_frac_231_max": "Pastures",
    "test_dyn_k_index_corine_frac_24_max": "Mixed agricultural",
    "test_dyn_k_index_corine_frac_31_max": "Forests",
    "test_dyn_k_index_corine_frac_322_max": "Moors and heathland",
    "test_dyn_k_index_corine_frac_32_max": "Scrub",
    "test_dyn_k_index_corine_frac_412_max": "Marshes, peat bogs",
    "test_dyn_k_index_corine_frac_4_max": "Wetlands",
    "test_dyn_k_index_corine_frac_5_max": "Water bodies",
    "test_dyn_k_index_meandist_road_max": "Distance to road",
    "test_dyn_k_index_pop_density_max": "Population density",
}

dict_descr = {
    "breezy-dew-64": "CLIP, soft loss, sigma=0.5",
    "stoic-glade-65": "CLIP, soft loss, sigma=1.0",
    "deep-plasma-67": "CLIP, soft loss, sigma=0.25",
    "light-oath-102": "CLIP, hard loss",
    "kind-surf-110": "LLM, soft loss, sigma=0.25",
}
# new_df = new_df.rename(columns=dict_name_scores)
new_df["experiment"] = new_df["run_name"].map(dict_descr)

new_df = new_df[new_df["run_name"].isin(runs_keep)]
new_df

In [ ]:
table_df = new_df.rename(columns=dict_name_scores)
table_df = table_df[["experiment"] + list(dict_name_scores.values())]

# make experiment new column names, and set other columns to index (transpose)
table_df = table_df.set_index("experiment")
table_df = table_df.T
for c in table_df.columns:
    if c == "experiment":
        continue
    table_df[c] = table_df[c].round(2)

drop_cols = ["CLIP, soft loss, sigma=0.5", "CLIP, soft loss, sigma=1.0"]
table_df = table_df.drop(columns=drop_cols)


table_df
# Make ready for latex table:
table_df.to_latex(
    "alignment_results.tex",
    index=True,
    float_format="%.2f",
    caption="Performance of the best models on the test set, averaged across seeds. The best model for each metric is highlighted in bold.",
    label="tab:alignment_models",
    escape=False,
)

In [ ]:
ax = plt.subplot(111)
sns.barplot(data=df, x="experiment", y="test_avr_top-dyn_k_index", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")

In [ ]:
scores_plot = [x for x in df.columns if x.startswith("test_dyn_k_index")]
dict_name_scores = {
    "test_dyn_k_index_bioclim_06_min": "Minimum temperature",
    "test_dyn_k_index_bioclim_12_max": "Precipitation",
    "test_dyn_k_index_corine_frac_11_max": "Urban fabric",
    "test_dyn_k_index_corine_frac_12_max": "Industrial areas",
    "test_dyn_k_index_corine_frac_21_max": "Arable land",
    "test_dyn_k_index_corine_frac_231_max": "Pastures",
    "test_dyn_k_index_corine_frac_24_max": "Mixed agricultural",
    "test_dyn_k_index_corine_frac_31_max": "Forests",
    "test_dyn_k_index_corine_frac_322_max": "Moors and heathland",
    "test_dyn_k_index_corine_frac_32_max": "Scrub",
    "test_dyn_k_index_corine_frac_412_max": "Marshes, peat bogs",
    "test_dyn_k_index_corine_frac_4_max": "Wetlands",
    "test_dyn_k_index_corine_frac_5_max": "Water bodies",
    "test_dyn_k_index_meandist_road_max": "Distance to road",
    "test_dyn_k_index_pop_density_max": "Population density",
}

assert set(scores_plot) == set(dict_name_scores.keys())


def plot_scores(
    df, experiment, scores_plot=scores_plot, plot_type="spider", ax=None, legend_name=None
):
    # colour_models_dict = {'aef': 'blue', 'geoclip': 'green', 'tessera': 'red'}
    # colour_plot = [v for k, v in colour_models_dict.items() if k in experiment][0]
    n_colours = len(df["experiment"].unique())
    colour_plot = plt.cm.get_cmap("tab10")(
        list(df["experiment"].unique()).index(experiment) / n_colours
    )
    if ax is None:
        ax = plt.subplot(111, polar=(plot_type == "spider"))
    df_plot = df[df["experiment"] == experiment]
    assert len(df_plot) > 0, f"No data for experiment {experiment}"
    scores = df_plot[scores_plot].mean(0)
    if plot_type == "spider":
        angles = np.linspace(0, 2 * np.pi, len(scores), endpoint=False).tolist()
        scores = np.concatenate((scores, [scores[0]]))
        angles += angles[:1]
        ax.plot(
            angles,
            scores,
            label=legend_name if legend_name is not None else experiment,
            c=colour_plot,
            linestyle=":" if "unlab" in experiment else "-",
        )
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([dict_name_scores[s] for s in scores_plot], rotation=45)
        ax.set_ylim([-0.1, 1])
    elif plot_type == "bar":
        ax.bar(scores_plot, scores)
        ax.set_xticklabels([dict_name_scores[s] for s in scores_plot], rotation=45, ha="right")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")


def print_scores(df, experiment, scores_plot=scores_plot):
    df_plot = df[df["experiment"] == experiment]
    assert len(df_plot) > 0
    scores = df_plot[scores_plot].mean(0)
    for s in scores_plot:
        print(f"{dict_name_scores[s]}: {scores[s]:.3f}")


fig, ax = plt.subplots(1, 1, figsize=(5, 5), subplot_kw={"polar": True})
for i_name, name in enumerate(new_df["experiment"].unique()):
    # if 'aef' not in name:
    #     continue
    # if i_name not in [1, 2, 4]:
    #     continue
    plot_scores(new_df, name, plot_type="spider", ax=ax, legend_name=name)
# plot_scores(df, 'alignment-geoclip-12345', plot_type='spider', ax=ax, legend_name='GeoCLIP')
# plot_scores(df, 'alignment-aef-aef_128-12345', plot_type='spider', ax=ax, legend_name='AEF')
# plot_scores(df, 'alignment-tessera-tessera_256-12345', plot_type='spider', ax=ax, legend_name='Tessera')

# print_scores(df, 'alignment-aef-aef_128-12345')

In [ ]:
df.experiment.unique()

In [ ]:
cols = [
    "test_dyn_k_index_corine_frac_5_max",
    "test_dyn_k_index_pop_density_max",
    "test_dyn_k_index_bioclim_12_max",
]
colour_models_dict = {"aef": "blue", "geoclip": "green", "tessera": "red"}

df_plot = df[df["experiment"].isin(["geoclip_cliptext_b256", "avr_aef_128_mlp_cliptext"])]
fig, ax = plt.subplots(len(cols), 1, figsize=(2, 3 * len(cols)), gridspec_kw={"hspace": 0.5})
for i, col in enumerate(cols):
    assert col in df.columns, f"{col} not in dataframe columns"
    sns.barplot(
        data=df_plot,
        x="experiment",
        y=col,
        ax=ax[i],
        palette=[colour_models_dict["geoclip"], colour_models_dict["aef"]],
    )
    ax[i].set_title(dict_name_scores[col])
    ax[i].set_ylim([0, 1])
    ax[i].set_ylabel("Top-k index")
    ax[i].set_xticklabels(["GeoCLIP", "AEF"])
    ax[i].set_xlabel("")
    # if i == len(cols) - 1:
    #     ax[i].set_xlabel('Model')

## Habitat preference caption inference

In [ ]:
# habitat_preferences = [
#     {'inference_captions': [
#         'Isolated shrub or scrub patches interspersed within open grassland matrix',
#         'Dense low-growing herbaceous vegetation with high spectral diversity indicating forb-rich grassland',
#         'Fine-scale mosaic of closely cropped and taller grassland patches'
#     ],
#     'common_name': 'Adonis blue',
#     'latin_name': 'Lysandra bellargus',
#     'target_id': 32
#     }
# ]

# import json
# with open('/Users/tplas/data/aether_data/s2bms/inference_captions/habitat_preferences.json', 'w') as f:
#     json.dump(habitat_preferences, f, indent=4)

In [ ]:
with open(
    "/Users/tplas/data/aether_data/s2bms/inference_captions/habitat_preferences.json", "r"
) as f:
    habitat_preferences_loaded = json.load(f)
# habitat_preferences_loaded

In [ ]:
for i_h, h in enumerate(habitat_preferences_loaded):
    print(
        f"Habitat {i_h}: {h['common_name']} ({h['latin_name']}), number of inference captions: {len(h['inference_captions'])}"
    )

In [ ]:
# path_inference_results = '/Users/tplas/data/aether_data/outputs/inference_results_2026-06-11-0906.pkl'
path_inference_results = (
    "/Users/tplas/data/aether_data/outputs/inference_results_2026-06-25-0821.pkl"
)
with open(path_inference_results, "rb") as f:
    inference_results = pickle.load(f)
inference_results = inference_results["test"]

# for k, v in inference_results.items():
#     if type(v) == dict:
#         print(f"{k}:")
#         for k2, v2 in v.items():
#             print(f"  {k2}: {v2.shape if hasattr(v2, 'shape') else type(v2)}")
#     else:
#         print(f"{k}: {v.shape}")

geo_embeddings = inference_results["geo_embeddings"]
inference_caption_embeddings = inference_results["inference_caption_embeddings"]
target_id_inference_captions = inference_results["target_id_inference_captions"]
labels = inference_results["labels"]
predictions = inference_results["predictions"]

# species_use = 'Adonis blue'
# assert species_use in target_id_inference_captions

mse_per_species = []
mean_rate_per_species = []
corr_sim_preds = []
corr_sim_labels = []
species_names = []

for species_use in target_id_inference_captions.keys():
    target_id_use = target_id_inference_captions[species_use]
    species_names.append(species_use)
    sim_matrix = np.dot(inference_caption_embeddings[species_use], geo_embeddings.T)
    sim_summed = sim_matrix.sum(0)

    mse_per_species.append(
        ((predictions[:, target_id_use] - labels[:, target_id_use]) ** 2).mean()
    )
    mean_rate_per_species.append(labels[:, target_id_use].mean())

    fig, ax = plt.subplots(1, 2, figsize=(8, 3))

    for i_ax, occ_plot in enumerate([labels, predictions]):
        ax[i_ax].scatter(sim_summed, occ_plot[:, target_id_use], alpha=0.5)
        corr = np.corrcoef(sim_summed, occ_plot[:, target_id_use])[0, 1]
        corr_sim_preds.append(corr) if i_ax == 1 else corr_sim_labels.append(corr)
        ax[i_ax].set_title(
            f"{species_use}\n {'True labels' if i_ax == 0 else 'Model predictions'} (corr={corr:.2f})"
        )
        ax[i_ax].set_xlabel("Summed similarity to inference captions")
        ax[i_ax].set_ylabel("Occurrence label" if i_ax == 0 else "Model prediction")

    miny = min(labels[:, target_id_use].min(), predictions[:, target_id_use].min())
    maxy = max(labels[:, target_id_use].max(), predictions[:, target_id_use].max())
    ax[0].set_ylim([miny, maxy])
    ax[1].set_ylim([miny, maxy])

In [ ]:
save_plot = True

# plot a pair plot of correlation values:
corr_sim_labels = np.squeeze(corr_sim_labels)
corr_sim_preds = np.squeeze(corr_sim_preds)
n = len(corr_sim_preds)


import scipy.stats as stats

inds_nonnan = ~np.isnan(corr_sim_labels) & ~np.isnan(corr_sim_preds)
ttest = stats.ttest_rel(
    corr_sim_labels[inds_nonnan], corr_sim_preds[inds_nonnan], alternative="less"
)
print(f"Paired t-test: t={ttest.statistic:.3f}, p={ttest.pvalue:.3f}")

fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.bar(
    np.arange(n) - 0.2,
    corr_sim_labels,
    width=0.4,
    label="True occurrence rates",
    color="blue",
    alpha=0.5,
    edgecolor="k",
)
ax.bar(
    np.arange(n) + 0.2,
    corr_sim_preds,
    width=0.4,
    label="Predicted occurrence rates",
    color="blue",
    edgecolor="k",
)
# ax.set_xlabel('Samples')
ax.set_ylabel(r"$\rho$" + "(embedding similarity,\noccurrence rates)")
ax.legend()
ax.set_xticks(np.arange(n))
ax.set_xticklabels(species_names, rotation=30, ha="right")

for sp in ["top", "right", "bottom"]:
    ax.spines[sp].set_visible(False)
# plt.show()

if save_plot:
    plt.savefig(
        f"../figs/embedding_similarity_correlation_{len(species_names)}species.pdf",
        dpi=300,
        bbox_inches="tight",
    )

# ax = plt.subplot(111)
# n = len(corr_sim_preds)
# x1 = np.random.randn(n) * 0.05
# x2 = np.random.randn(n) * 0.05 + 1
# ax.scatter(x1, corr_sim_labels)
# ax.scatter(x2, corr_sim_preds)
# for k in range(n):
#     ax.plot([x1[k], x2[k]], [corr_sim_labels[k], corr_sim_preds[k]], c='gray', alpha=0.5)
# ax.set_xticks([0, 1])

In [ ]:
corr_sim_preds